Path to your workspace

In [ ]:
ws = '/path/to/workspace'

Bash function

In [ ]:
#### Bash function
def bash(string, name, queue):
  script = "%s.sh" % name
  with open(script, "w") as text_file:
    text_file.write(string)
  !chmod +x $script
  !qsub -q $queue $script
  #!rm $script

Make folders for the workflow

In [ ]:
mkdir fasta raw ft regular sip configs sipros

#fasta <--- fasta files
#raw <--- raw data i.e. mass spectrometry data
#ft <--- will contain files after Step 1
#regular <--- will contain output files from unlabeled search
#sip <--- will contain files after labeled search  
#configs <--- will contain config files for each batch
#sipros <--- contains sipros configuration files

Step 1: Convert RAW → FT

In [ ]:
string = '''
#PBS -l nodes=1:ppn=8
#PBS -l walltime=24:00:00
#PBS -l mem=64gb
#PBS -q short
#PBS -S /bin/bash

source /home/path/to/conda.sh
conda activate SIPROS

mono /path/to/sipros/bin/Raxport.exe -i %s -o %s -j 8
'''

input = ws + 'path/to/raw'
output = ws + 'path/ft'

bash(string % (input, output), 'convert_raw', 'short')

Step 2: Define samples

In [ ]:
### put the name of your samples here instead of the numbers

#### No MI samples (triplicates i.e. each row represent samples from one batch)

samples_noMI = [
    '217','218','219',
    '226','227','228',
    '235','236','237',
    '244','245','246',
    '253','254','255'
]

#### 12C MI samples (triplicates i.e. each row represent samples from one batch)
samples_12C = [
    '220','221','222',
    '229','230','231',
    '238','239','240',
    '247','248','249',
    '256','257','258'
]

Step 3: Make fasta database with reverse decoy

In [ ]:
string = '''
#PBS -l nodes=1:ppn=14
#PBS -l walltime=12:00:00
#PBS -l mem=64gb
#PBS -q short
#PBS -S /bin/bash

source /path/to/conda.sh
conda activate SIPROS

# make reverse
python /path/to/sipros/EnsembleScripts/sipros_prepare_protein_database.py -i %s -o %s -c /path/to/sipros/configTemplates/SiprosEnsembleConfig.cfg
'''

input = ws + 'fasta/protein_catalogue_cRAP.fasta'   # put the name of your protein fasta file here
output = ws + 'fasta/Decoy.fasta'

bash(string%(input, output), 'fasta', 'short')

Step 4: Unlabeled search

In [ ]:
#### search FT2 scans of unlabeled samples against fasta DB

string = '''
#PBS -l nodes=1:ppn=8
#PBS -l walltime=24:00:00
#PBS -l mem=32gb
#PBS -q long
#PBS -S /bin/bash

source /path/to/conda.sh
conda activate SIPROS

export OMP_NUM_THREADS=8

mkdir -p %s

/path/to/sipros/bin/SiprosEnsembleOMP -f %s -c /path/to/sipros/configTemplates/SiprosEnsembleConfig.cfg -o %s
'''

for s in samples_12C:
    
    output_folder = ws + 'regular/' + s
    input_file = ws + f'ft/{s}.FT2'
    output_directory = ws + 'regular/' + s
    bash(string % (output_folder, input_file, output_directory), f'scandb{s}', 'long')

Step 5: Post-processing

In [ ]:
#### PSM filtering, protein assembly, FDR, spectral count

string = '''
#PBS -l nodes=1:ppn=8
#PBS -l walltime=24:00:00
#PBS -l mem=64gb
#PBS -q long
#PBS -S /bin/bash

# For py2
source /path/to/conda.sh
conda activate SIPROS

# For R
source /path/to/conda.sh
conda activate SIPROS

# convert .Spe2Pep.txt file to .tab file
python /path/to/sipros/EnsembleScripts/sipros_psm_tabulating.py -i %s -o %s

# filter PSMs, output qualified PSMs to *.psm.txt file
python /path/to/sipros/EnsembleScripts/sipros_ensemble_filtering.py -i %s -o %s

# assemble protein groups from peptide, output proteins to *.pro.txt file
python /path/to/sipros/EnsembleScripts/sipros_peptides_assembling.py -w %s

# control FDR, output qualified protein groups to *.proRefineFDR.txt file
Rscript /path/to/sipros/V4Scripts/refineProteinFDR.R -pro %s/*.pro.txt -psm %s/*.psm.txt -fdr 0.005 -o %s

# get spectra count of each protein groups, output spectra count to .SPcount.txt
Rscript /path/to/sipros/V4Scripts/getSpectraCountInEachFT.R -pro %s/*.proRefineFDR.txt -psm %s/*.psm.txt -o %s
'''

for s in samples_12C:
    input1 = ws + 'regular/' +s
    output1 = ws + 'regular/' +s
    input2 = ws + 'regular/' +s
    output2 = ws + 'regular/' +s
    output3 = ws + 'regular/' +s
    input3 = ws + f'regular/{s}/*.pro.txt'
    input4 = ws + f'regular/{s}/*.psm.txt'
    output4 = ws + f'regular/{s}/Sample_{s}'
    input5 = ws + f'regular/{s}/*.proRefineFDR.txt'
    input6 = ws + f'regular/{s}/*.psm.txt'
    output5 = ws + f'regular/{s}/Sample_{s}'

    bash(string%(input1, output1, input2, output2, output3, input3, input4, output4, input5, input6, output5 ), f'Unlabeled{s}', 'long')

Step 6: Select reference samples

In [ ]:
### put the name of your selected reference samples here instead of the numbers
#### Selected 12C samples (based on highest counts of identifications)

reference_samples = ['220','230','239','247','256']

Step 7: Build reduced database

In [ ]:
#### create database (DB) from identified proteins

string = '''
#PBS -l nodes=1:ppn=8
#PBS -l walltime=1:00:00
#PBS -l mem=64gb
#PBS -q short
#PBS -S /bin/bash

source /path/to/conda.sh
conda activate SIPROS

# Create output directory if not already there
mkdir -p %s

Rscript /path/to/sipros/V4Scripts/makeDBforLabelSearch-Copy.R -pro %s -faa %s -o %s
'''
# Since the reference samples are different per group, hence different DB created per group i.e. inside regular folder, diffreent subfolders named after reference_samples will be containing the DB for each group

for s in reference_samples:

    output_dir = ws + f'regular/{s}/fasta'
    input_pro = ws + f'regular/{s}/*.SE.pro.txt'
    fasta = ws + 'fasta/protein_catalogue_mgnify+cRAP.fasta'
    output_db = ws + f'regular/{s}/fasta/db.faa'

    bash(string % (output_dir, input_pro, fasta, output_db), f'db_{s}', 'short')

Step 8: Create Config files

In [ ]:
string = '''
#PBS -l nodes=1:ppn=2
#PBS -l walltime=1:00:00
#PBS -l mem=10gb
#PBS -q short
#PBS -S /bin/bash

/path/to/sipros/bin/configGenerator -i %s -o %s -e C
'''
# Diffreent config files created per sample

for s1 in reference_samples:
    input_file = ws + f'sipros/configTemplates/SiprosV4Config-{s1}.cfg'
    output= ws + f'regular/{s1}/configs'
    
    bash(string%(input_file, output), f'configs{s1}', 'short')

Step 9: Define groups

In [ ]:
### put the name of your samples here instead of the numbers e.g. 'reference sample': ['No MI control sample', '12C control sample', '13C sample']
#### Groups = same hen + section + timepoint

batches = {
    '220': ['217','218','219','220','221','222','223','224','225'],
    '230': ['226','227','228','229','230','231','232','233','234'],
    '239': ['235','236','237','238','239','240','241','242','243'],
    '247': ['244','245','246','247','248','249','250','251','252'],
    '256': ['253','254','255','256','257','258','259','260','261']
}

Step 10: Labeled search

In [ ]:
#### labeled search using group-specific DB

string = '''
#PBS -l nodes=1:ppn=8
#PBS -l walltime=24:00:00
#PBS -l mem=64gb
#PBS -q long
#PBS -S /bin/bash

source /path/to/conda.sh
conda activate SIPROS

export OMP_NUM_THREADS=8

# Create output directory if not already there
mkdir -p %s

#Define the configuration file
configs=(%s)

# Process each config file in parallel with limit of 8 processes and all insert the location of database
cd /path/to/folder
echo "${configs[@]}" | xargs -n 1 -P 8 \
        bash -c '/path/to/sipros/bin/SiprosV4OMP -f %s -c $0 -o %s'

echo "filter PSMs"
python /path/to/sipros/V4Scripts/sipros_peptides_filtering.py -c /path/to/sipros/configTemplates/SiprosV4Config-%s.cfg -w %s

echo "filter proteins"
python /path/to/sipros/V4Scripts/sipros_peptides_assembling.py -c /path/to/sipros/configTemplates/SiprosV4Config-%s.cfg -w %s

echo "cluster SIP abundance of protein"
python /path/to/sipros/V4Scripts/ClusterSip.py -c /path/to/sipros/configTemplates/SiprosV4Config-%s.cfg -w %s

echo "refine protein FDR"
Rscript /path/to/sipros/V4Scripts/refineProteinFDR.R -pro %s -psm %s -fdr 0.01 -o %s

echo "get SIP abundance of each protein in each FT2 file or raw file. -thr set the threshold of isotopic atom"
Rscript /path/to/sipros/V4Scripts/getLabelPCTinEachFT.R -pro %s -psm %s -thr 5 -o %s

'''

for k, v in batches.items():
    config = '/path/to/sipros/configTemplates/SiprosV4Config.cfg'
    new = f'/path/to/sipros/configTemplates/SiprosV4Config-{k}.cfg'
    with open(config, 'r') as conf, open(new, 'w') as newconf:
        txt = conf.read()
        prefix = txt.split('FASTA_Database = /')[0]
        suffix = txt.split('/fasta/db.faa')[-1]
        middle = f'FASTA_Database = /path/to/regular/{k}/fasta/db.faa'
        newconf.write(prefix +'\n'+ middle +'\n'+ suffix)
                           
    for s1 in v:
        
        output_dir = ws + f'sip/{s1}'
        input_config = ws + f'regular/{k}/configs/*.cfg'
        input1 = ws + f'ft/Harshita_{s1}.FT2'
        output1 = ws + 'sip/' +s1
        output2 = ws + 'sip/' +s1
        output3 = ws + 'sip/' +s1
        output4 = ws + 'sip/' +s1
        input2 = ws + f'sip/{s1}/*.pro.txt'
        input3 = ws + f'sip/{s1}/*.psm.txt'
        output5 = ws + f'sip/{s1}/Sample_{s1}'
        input4 =  ws + f'sip/{s1}/*.proRefineFDR.txt'
        input5 = ws + f'sip/{s1}/*.psm.txt' 
        output6 = ws + f'sip/{s1}/Sample_{s1}'
        
        bash(string%(output_dir, input_config, input1, output1, k, output2, k, output3, k, output4, input2, input3, output5, input4, input5, output6 ), f'Label{s1}', 'long')